In [3]:
# Setup
import sys
sys.path.append('..')

from src.multi_file_loader import MultiFileLoader
from src.context_preserver import ContextPreserver
from src.summarizer import Summarizer
from src.vector_store import VectorStore
from src.config import Config
from pathlib import Path

from tqdm import tqdm
print("✓ Imports successful")

✓ Imports successful


## Step 1: Initialize Services

In [5]:
# Load configuration
config = Config()

# Check Qdrant
print(f"Qdrant URL: {config.qdrant_url}")
print(f"Collection: {config.qdrant_collection}")

# Check API access
if Config.AZURE_OPENAI_ENDPOINT:
    print(f"✓ Azure OpenAI: {Config.AZURE_OPENAI_ENDPOINT}")
    print(f"  Embedding deployment: {Config.AZURE_OPENAI_EMBEDDING_DEPLOYMENT}")
elif config.openai_api_key:
    print("✓ OpenAI API key loaded")
else:
    print("⚠ No API key configured")


Qdrant URL: http://localhost:6333
Collection: confidential_docs
✓ Azure OpenAI: https://initiative.openai.azure.com/
  Embedding deployment: text-embedding-3-large


In [6]:
# Initialize vector store with Azure OpenAI embeddings (text-embedding-3-large, 3072 dims)
if Config.AZURE_OPENAI_ENDPOINT:
    vector_store = VectorStore(
        qdrant_url=config.qdrant_url,
        collection_name=config.qdrant_collection,
        openai_api_key=Config.AZURE_OPENAI_API_KEY,
        azure_endpoint=Config.AZURE_OPENAI_ENDPOINT,
        api_version=Config.AZURE_OPENAI_API_VERSION,
        embedding_deployment=Config.AZURE_OPENAI_EMBEDDING_DEPLOYMENT  # text-embedding-3-large
    )
else:
    vector_store = VectorStore(
        qdrant_url=config.qdrant_url,
        collection_name=config.qdrant_collection,
        openai_api_key=Config.OPENAI_API_KEY
    )

# Health check
health = vector_store.health_check()
print(f"✓ Qdrant connection: {health['status']}")
print(f"  Embedding model: {vector_store.embedding_model} ({vector_store.embedding_dimensions} dims)")


✓ Qdrant connection: healthy
  Embedding model: text-embedding-3-large (3072 dims)


In [7]:
# Initialize Summarizer for image description (GPT-4.1 has built-in vision)
if Config.AZURE_OPENAI_ENDPOINT:
    summarizer = Summarizer(
        openai_api_key=Config.AZURE_OPENAI_API_KEY,
        azure_endpoint=Config.AZURE_OPENAI_ENDPOINT,
        api_version=Config.AZURE_OPENAI_API_VERSION,
        azure_deployment=Config.AZURE_OPENAI_CHAT_DEPLOYMENT  # gpt-4.1 supports vision
    )
else:
    summarizer = Summarizer(
        openai_api_key=config.openai_api_key,
        model="gpt-4.1"
    )

print(f"✓ Summarizer initialized (model: {summarizer.model})")
print(f"  Vision model: {summarizer.vision_model}")
print(f"  Will be used for: image description only (text/code skip summarization)")

✓ Summarizer initialized (model: gpt-4.1)
  Vision model: gpt-4.1
  Will be used for: image description only (text/code skip summarization)


## Step 2: Create/Recreate Collection

In [ ]:
# Create collection — dimensions auto-detected from embedding model
# text-embedding-3-large = 3072 dims | text-embedding-3-small = 1536 dims
vector_store.create_collection(
    recreate=True  # Set to False to keep existing data
)

print(f"✓ Collection '{config.qdrant_collection}' created/verified")
print(f"  Dimensions: {vector_store.embedding_dimensions}")


## Step 3: Prepare Elements

In [ ]:
print("Phase 1: Loading documents...")
loader = MultiFileLoader()
documents = loader.load_all_documents(str(data_folder))
print(f"✓ Loaded {len(documents)} documents")

In [ ]:
import pandas as pd
documents_df = pd.DataFrame(documents)

In [ ]:
def get_summary_from_summarizer(doc):
    try:
        if doc.get('type') == 'image':
            return summarizer.describe_image(
                image_base64=doc['image_data'],
                context=f"File: {doc.get('filename', 'unknown')} from FLOWCAL documentation"
            )
        else:
            return doc.get('summary_short', '')
    except Exception as e:
        print(f"Error summarizing result: {e}")
        return ''

In [ ]:
from tqdm import tqdm

def process_row(idx_row):
    idx, row = idx_row
    return idx, get_summary_from_summarizer(row)

rows = list(documents_df.iterrows())
results = [None] * len(rows)

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(process_row, row): row[0] for row in rows}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Summarizing"):
        idx, summary = future.result()
        results[idx] = summary

documents_df['describe_summary'] = results

In [ ]:
documents_df['describe_summary'][3]

In [ ]:
image_docs_df = documents_df[documents_df['describe_summary']!=''].reset_index(drop=True)

In [ ]:
image_docs_df['image_description'] = image_docs_df['describe_summary'].apply(lambda col: col.get('full', ''))
image_docs_df['image_description_short'] = image_docs_df['describe_summary'].apply(lambda col: col.get('short', ''))
# described += 1

In [ ]:
image_docs_df[['filename', 'describe_summary', 'image_description', 'image_description_short']].head()

In [ ]:
image_docs_df.to_csv('image_docs_with_descriptions.csv', index=False)

In [ ]:
sample_image_docs_df = pd.read_csv('image_docs_with_descriptions.csv')

In [ ]:
documents_df.to_csv('all_docs_with_descriptions.csv', index=False)

In [ ]:
import warnings
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
warnings.filterwarnings('ignore', category=UserWarning)

# Load, describe images, link, and prepare documents for vector storage
data_folder = Path("../data")

if not data_folder.exists():
    print("⚠ No data folder found. Please add documents to ../data/")
    elements = []
else:
    # === Phase 1: Load all documents ===
    print("Phase 1: Loading documents...")
    loader = MultiFileLoader()
    documents = loader.load_all_documents(str(data_folder))
    print(f"✓ Loaded {len(documents)} documents")
    
    # === Phase 2: Describe standalone images with AI vision (10 threads) ===
    print("\nPhase 2: Describing images with AI vision (10 threads)...")
    image_docs = [d for d in documents if d.get('type') == 'image' and 'image_data' in d]
    print(f"  Found {len(image_docs)} standalone images to describe")
    
    described = 0
    failed = 0
    start_time = time.time()

    def describe_single_image(idx_doc):
        idx, doc = idx_doc
        try:
            desc = summarizer.describe_image(
                image_base64=doc['image_data'],
                context=f"File: {doc.get('filename', 'unknown')} from FLOWCAL documentation"
            )
            return idx, doc, desc, None
        except Exception as e:
            return idx, doc, None, e

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {
            executor.submit(describe_single_image, (idx, doc)): idx
            for idx, doc in enumerate(image_docs)
        }
        for future in tqdm(as_completed(futures), total=len(futures), desc="Describing images"):
            idx, doc, desc, error = future.result()
            if error is None:
                doc['image_description'] = desc.get('full', '')
                doc['image_description_short'] = desc.get('short', '')
                described += 1
            else:
                doc['image_description'] = f"Image: {doc.get('filename', 'unknown')}"
                doc['image_description_short'] = doc.get('filename', 'Image')
                failed += 1
                if failed <= 3:
                    print(f"  ⚠ Failed: {doc.get('filename')}: {error}")
    
    elapsed = time.time() - start_time
    print(f"✓ Described {described}/{len(image_docs)} images ({failed} failed) in {elapsed/60:.1f} min")
    print(f"  API calls: {summarizer.get_statistics()['total_api_calls']}")
    
    # === Phase 3: Link elements with context ===
    print("\nPhase 3: Linking elements...")
    preserver = ContextPreserver()
    for doc in documents:
        # Normalize: some doc types use 'content' instead of 'text_chunks'
        text_chunks = doc.get('text_chunks', [])
        if not text_chunks and 'content' in doc:
            text_chunks = [{
                'element_id': f"{doc.get('file_id', 'unknown')}_text_0",
                'content': doc['content'],
                'type': 'text'
            }]
        
        # Handle standalone image docs — use AI description as searchable content
        images = doc.get('images', [])
        if doc.get('type') == 'image' and 'image_data' in doc:
            description = doc.get('image_description', f"Image: {doc.get('filename', 'unknown')}")
            short_desc = doc.get('image_description_short', doc.get('filename', 'Image'))
            images = [{
                'element_id': f"{doc.get('file_id', 'unknown')}_img_0",
                'content': description,       # Full AI description — used for embedding
                'summary_short': short_desc,   # Short description — stored in payload
                'summary_medium': description, # Medium = full for images
                'type': 'image'
            }]
        
        if not text_chunks and not doc.get('tables', []) and not images:
            continue
        preserver.add_document_elements(
            file_id=doc.get('file_id', ''),
            filename=doc.get('filename', ''),
            text_chunks=text_chunks,
            tables=doc.get('tables', []),
            images=images
        )
    elements = preserver.all_elements
    
    # Count types
    type_counts = {}
    for e in elements:
        t = e.get('type', 'unknown')
        type_counts[t] = type_counts.get(t, 0) + 1
    
    print(f"✓ Linked {len(elements)} elements — ready for embedding and upload")
    print(f"  Breakdown: {type_counts}")

## Step 4: Generate Embeddings and Upload

In [ ]:
import logging
logging.basicConfig(level=logging.ERROR, format='%(levelname)s - %(message)s')

# Upload elements to Qdrant
if elements:
    print(f"\nUploading {len(elements)} elements to Qdrant...")
    
    result = vector_store.upsert_elements(
        elements=elements,
        batch_size=100  # Process 100 at a time
    )
    
    print(f"\n✓ Upload complete!")
    print(f"  Successful: {result['successful']}")
    print(f"  Failed: {result['failed']}")
    print(f"  Total embeddings generated: {result['embeddings_generated']}")
else:
    print("⚠ No elements to upload")

In [ ]:
print(result)

## Step 5: Basic Similarity Search

In [8]:
# Test search
query = "What is the main purpose of this document?"

print(f"Query: {query}\n")
print("=" * 80)

results = vector_store.search(
    query=query,
    top_k=5
)

print(f"\nFound {len(results)} results:\n")

for idx, result in enumerate(results, 1):
    print(f"{idx}. Score: {result['score']:.3f}")
    print(f"   File: {result['filename']}")
    print(f"   Type: {result['type']}")
    print(f"   Path: {result.get('hierarchy_path', 'N/A')}")
    
    # Show content — prefer summary, fall back to content
    snippet_source = result.get('summary_medium') or result.get('summary_short') or result.get('content', '')
    if snippet_source:
        snippet = snippet_source[:200] + "..." if len(snippet_source) > 200 else snippet_source
        print(f"   Content: {snippet}")
    
    print()

Query: What is the main purpose of this document?



Embedding batches: 100%|██████████| 1/1 [00:02<00:00,  2.46s/it]


Found 5 results:

1. Score: 0.400
   File: Manual Performance Test Template Instructions.docx
   Type: text
   Path: Manual Performance Test Template Instructions.docx
   Content: The purpose of this section is to collect information on any prerequisites and additionally setup prior to starting the data collection process. 

2. Score: 0.391
   File: 05.01.02 Master Data Creation.docx
   Type: text
   Path: 05.01.02 Master Data Creation.docx
   Content: Supporting Document Location

3. Score: 0.377
   File: document icon-05_gray_14x17.png
   Type: image
   Path: document icon-05_gray_14x17.png
   Content: Certainly! Here is a detailed description of the image:

---

### 1. What the image shows
The image is a small, gray icon representing a document. It is commonly used in software interfaces to symboli...

4. Score: 0.367
   File: document icon-05_gray.png
   Type: image
   Path: document icon-05_gray.png
   Content: Certainly! Here is a detailed description of the image:

---

### 1. 

## Step 6: Search with Relationships

In [9]:
# Search and retrieve related elements
query = "Show me information about tables and charts"

print(f"Query: {query}\n")
print("=" * 80)

search_result = vector_store.search_with_relationships(
    query=query,
    top_k=3,
    include_related=True
)

main_results = search_result['main_results']
related_elements = search_result['related_elements']

print(f"\nMain results: {len(main_results)}")
print(f"Related elements: {len(related_elements)}")

# Show main results
print("\n--- Main Results ---\n")
for idx, result in enumerate(main_results, 1):
    print(f"{idx}. {result['type']} from {result['filename']}")
    print(f"   Score: {result['score']:.3f}")
    print(f"   Related: {len(result.get('related_text_ids', []))} text, "
          f"{len(result.get('related_image_ids', []))} images, "
          f"{len(result.get('related_table_ids', []))} tables")
    print()

# Show related elements
if related_elements:
    print("\n--- Related Elements ---\n")
    for idx, elem in enumerate(related_elements[:5], 1):  # Show first 5
        print(f"{idx}. {elem['type']} from {elem['filename']}")
        summary = elem.get('summary_short', '')
        if summary:
            print(f"   {summary[:100]}..." if len(summary) > 100 else f"   {summary}")
        print()

Query: Show me information about tables and charts



Embedding batches: 100%|██████████| 1/1 [00:01<00:00,  1.38s/it]



Main results: 3
Related elements: 10

--- Main Results ---

1. text from Reports.htm
   Score: 0.422
   Related: 0 text, 0 images, 0 tables

2. text from 07.02.06.01.13-.14 - Tank Creation.docx
   Score: 0.415
   Related: 5 text, 0 images, 0 tables

3. text from 07.05.03.01-.04 - Tank Creation.docx
   Score: 0.415
   Related: 5 text, 0 images, 0 tables


--- Related Elements ---

1. text from 07.02.06.01.13-.14 - Tank Creation.docx

2. text from 07.02.06.01.13-.14 - Tank Creation.docx

3. text from 07.05.03.01-.04 - Tank Creation.docx

4. text from 07.05.03.01-.04 - Tank Creation.docx

5. text from 07.02.06.01.13-.14 - Tank Creation.docx



## Step 7: Filtered Search

In [10]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

# Search only in images
query = "diagram or chart"

print(f"Query: {query}")
print("Filter: type = 'image'\n")
print("=" * 80)

filter_condition = Filter(
    must=[
        FieldCondition(
            key="type",
            match=MatchValue(value="image")
        )
    ]
)

results = vector_store.search(
    query=query,
    top_k=5,
    filter_by=filter_condition
)

print(f"\nFound {len(results)} image results:\n")

for idx, result in enumerate(results, 1):
    print(f"{idx}. Score: {result['score']:.3f}")
    print(f"   File: {result['filename']}")
    print(f"   Page: {result.get('page_number', 'N/A')}")
    print(f"   Path: {result.get('hierarchy_path', 'N/A')}")
    
    # Show image description
    desc = result.get('summary_short', '')
    if desc:
        print(f"   Description: {desc}")
    print()

Query: diagram or chart
Filter: type = 'image'



Embedding batches: 100%|██████████| 1/1 [00:01<00:00,  1.29s/it]


Found 5 image results:

1. Score: 0.459
   File: Legend Alarm State Codes - 8.11.png
   Page: None
   Path: Legend Alarm State Codes - 8.11.png

2. Score: 0.453
   File: Alarm State Codes.png
   Page: None
   Path: Alarm State Codes.png

3. Score: 0.449
   File: Ticket Validation Flow Chart_635x300.png
   Page: None
   Path: Ticket Validation Flow Chart_635x300.png

4. Score: 0.448
   File: Index Values Flowchart.png
   Page: None
   Path: Index Values Flowchart.png

5. Score: 0.445
   File: Index Values Flowchart_555x938.png
   Page: None
   Path: Index Values Flowchart_555x938.png



## Step 8: Collection Statistics

In [11]:
# Get collection stats
stats = vector_store.get_statistics()

print("Collection Statistics:")
print("=" * 80)
print(f"\nCollection: {stats.get('collection_name', 'N/A')}")
print(f"Status: {stats.get('status', 'N/A')}")
print(f"\nPoints (elements): {stats.get('points_count', 0):,}")
print(f"Vectors stored: {stats.get('vectors_count', 'N/A')}")

if 'vector_size' in stats:
    print(f"Vector dimensions: {stats['vector_size']}")

if 'distance_metric' in stats:
    print(f"Distance metric: {stats['distance_metric']}")

print("\n" + "=" * 80)
print("\n✓ Vector storage demonstration complete!")
print("\nNext: 05_rag_system.ipynb - Full RAG query pipeline")

Collection Statistics:

Collection: confidential_docs
Status: green

Points (elements): 124,776
Vectors stored: 124776


✓ Vector storage demonstration complete!

Next: 05_rag_system.ipynb - Full RAG query pipeline
